# LKIPA Time Series Data Collection

In this notebook we develop the script to collect time series data from the LKIPA. 

- PUMP OFF
- PUMP ON (Sub-Threshold)
- PUMP ON (Threshold)
- PUMP ON (Super-Threshold)

PUMP OFF data is required to quantify the background noise level and frequency spectrum to subtract from the PUMP ON cases. 

----

In [4]:
# IMPORTS
# =======
import os
import sys

import matplotlib.pyplot as plt
#%matplotlib widget
import numpy as np
from tqdm import tqdm
from IPython.display import display, clear_output
import time

from presto import test
from presto import lockin, utils
from presto.hardware import AdcMode, DacMode, AdcFSample, DacFSample

import importlib
import LKIPA_library as lib
importlib.reload(lib)

<module 'LKIPA_library' from '/home/nanophys-meas/Desktop/Jai Master Thesis/Presto-Measurement-Scripts/LKIPA Time-Frequency Analysis/LKIPA_library.py'>

## 1. PRESTO Configuration

In [5]:
###### ------MEASUREMENT CONFIGURATION------ ######

# Sampling resolution
dt = 1e-9

# FFT resolution
DF = 100e3                                      # Hz

# Pump power sweep list (FS)
PUMP_AMP_LIST = [0.8] #np.arange(0, 1.0, 0.2)    # amplitude of pump signal, 0 for vacuum

# DC bias (V)
DC_BIAS = 0.0 

# Resonator frequency (Hz)
RES_FREQ = 4.4265e9                             # Set based on latest PSD/Covariance measurements, Hz

# Pump frequency for PRESTO  (Hz)
PUMP_FREQ = 2 * RES_FREQ - lib.PUMP_NCO         # Hz, 0 to 500 MHz, intermediate frequency

# Number of pixels to be captured
N_PIX = 5000            

### Notes

- dcb 3.5, f0 4.4242

## 2. Data Acquisition

In [6]:
# DATA ACQUISITION

# Filename based on timestamp of experimental run
Ym_str = time.strftime("%Y-%m")                                 # Get year and month
meas_type   = 'Time series'                                     # Measurement type
save_folder = r'/media/nanophys-meas/DR_BACKUP/Jai LKIPA Data/{}/{}'.format(Ym_str, meas_type)     # Create folder for current month and measurement type
myrun       = time.strftime("%Y-%m-%d_%H_%M_%S")                # Save experimental run for each timestamp
save_file   = r"{}.hdf5".format(myrun)                          # Save data in hdf5 file for current run


lib.data_acquisition(
    address=lib.ADDRESS,
    port=lib.PORT,
    converter_configuration=lib.CONVERTER_CONFIGURATION,
    input_port=lib.INPUT_PORT,
    adc_att=lib.ADC_ATT,
    input_nco=RES_FREQ,  # Set input NCO to resonator frequency to directly capture the downconverted signal at baseband
    output_port=lib.FLUX_PORT,
    dac_curr=lib.DAC_CURR,
    amp_list=PUMP_AMP_LIST,
    freq=PUMP_FREQ,
    phasei=lib.PHASEI,
    phaseq=lib.PHASEQ,
    output_nco=lib.PUMP_NCO,
    df=DF,
    dcb_port=lib.DC_PORT,
    dcb_amp=DC_BIAS,
    n_pix=N_PIX,
    myrun=myrun,
    folder=save_folder,
    file=save_file
)

Presto outputs reset to 0, data successfully save to 2026-07-21_08_22_08.hdf5.

MEASUREMENT PARAMETERS:
Mode: AdcMode.Mixed
Number of pixels: 5000
Pixel time resolution (dt): 1.00 ns
Sampling frequency (fs): 1.00 GHz
Total measurement time: 10.0 µs
Frequency resolution (DF): 100.0 kHz
